# Sterowanie trajektorią w modelach generatywnych

![Kot i dyfuzja](img/img.png)

W modelach generatywnych (dyfuzyjnych i opartych na algorytmach typu *flow-matching*) obraz nie powstaje w jednym kroku. Proces ten określa **trajektoria** – sekwencja stanów prowadząca od początkowego *szumu* $x_T$ do gotowego obrazu $x_0$. Każdy *sampling step* aktualizuje dane, bezpośrednio decydując o ostatecznym wyglądzie grafiki.

## Przebieg procesu (Pipeline)

Trajektoria fizycznie realizuje się poprzez wieloetapowy *denoising*. Standardowy schemat tego procesu (*pipeline*) wygląda następująco:

1. **Inicjalizacja:** Proces startuje od losowego *Gaussian noise* $x_T \sim \mathcal{N}(0, \mathbf{I})$. W zadaniach edycyjnych (*image-to-image*) wejściem jest oryginalny obraz zaszumiony do ustalonego poziomu $t < T$.
2. **Conditioning:** Model przyjmuje instrukcję $c$ (np. *embedding* promptu z *text encodera*), która wyznacza kierunek *denoisingu* w rozkładzie $p_\theta(x_{t-1} | x_t, c)$.
3. **Denoising loop:** Sieć neuronowa $\epsilon_\theta$ estymuje ilość *noise'u* w danych $x_t$, a dedykowany *scheduler* go redukuje. Powstaje ciąg stanów $\{x_T, x_{T-1}, \dots, x_0\}$, tworzący fizyczny kształt trajektorii.
4. **Decoding:** Po zakończeniu pętli, *decoder* (np. model VAE) zamienia skompresowane dane z *latent space* na finalne piksele obrazu.

## Wpływ trajektorii na obraz

Trajektoria wyznacza stany pośrednie, co wprost decyduje o procesie powstawania obrazu:

* **Ogólny zarys:** W początkowych krokach (dla dużych $t$) trajektoria ustala kompozycję, proporcje i układ głównych elementów sceny.
* **Tworzenie detali:** W krokach końcowych ($t \to 0$) algorytm generuje drobne szczegóły. Inna trajektoria przy tym samym wejściowym *noise* $x_T$ i warunku $c$ wygeneruje inne tekstury (np. inny układ porów na skórze czy refleksów światła).
* **Conditioning strength:** Określa, jak mocno prompt dyktuje ostateczny wynik. W fazie *inference time* steruje tym parametr *CFG Scale*. Algorytm wytycza w tle dwie równoległe trajektorie: jedną podyktowaną promptem i drugą całkowicie losową. *CFG Scale* to siła, która odpycha obraz od losowości, wymuszając ruch w stronę Twoich wytycznych. Im wyższa wartość, tym ślepiej model słucha instrukcji kosztem artystycznej swobody, jednak zbyt wyśrubowane CFG skutkuje utratą naturalności i "przepaleniem" kolorów.

## Kształt trajektorii

Z punktu widzenia inżynierii obliczeniowej, trajektoria jest zawsze realizowana w sekwencji dyskretnych kroków. O wydajności i precyzji modelu decyduje jednak fizyczny **kształt ścieżki** łączącej początkowy *noise* z docelowym obrazem.

Rozróżniamy dwa główne podejścia do kształtowania trajektorii:

* **Curved trajectories (nieliniowe):** W klasycznych modelach dyfuzyjnych naturalna ścieżka *denoisingu* meandruje przez złożone rozkłady danych. Algorytm musi nieustannie korygować kierunek, co wymusza podział trasy na wiele krótkich kroków (zazwyczaj 30-100). Skutkuje to wysokim kosztem obliczeniowym, długim *inference time* oraz ryzykiem powstawania wizualnych artefaktów na ostrych załamaniach trajektorii.
* **Rectified trajectories (liniowe):** Nowoczesne architektury (*Rectified Flow*, *Flow Matching*) wprowadzają matematyczne prostowanie wektorów. Algorytm wytycza najkrótszą, liniową ścieżkę w *state space* między szumem $x_T$ a obrazem $x_0$. Zmniejsza to ryzyko błędów i pozwala zredukować proces do zaledwie kilku długich kroków (często 1-4). Efektem jest błyskawiczny *inference time* i ścisła spójność z promptem, gdyż model zmierza bezpośrednio do celu.

## Co można osiągnąć za pomocą sterowania trajektorią?

Świadome manipulowanie przebiegiem trajektorii to fundament zaawansowanej pracy z generatywną sztuczną inteligencją. Pozwala ono na osiągnięcie konkretnych rezultatów inżynieryjnych i wizualnych:

* **Few-step generation:** Zastosowanie modeli o wyprostowanych trajektoriach pozwala zredukować liczbę *sampling steps* z kilkudziesięciu do kilku, generując obrazy w czasie rzeczywistym przy minimalnym zużyciu zasobów obliczeniowych (VRAM).
* **Inpainting / Image-to-image:** Manipulując długością trajektorii, decydujemy o stopniu modyfikacji obrazu. Krótka trajektoria zachowuje oryginalną kompozycję i tożsamość obiektu (zmieniając np. tylko oświetlenie), podczas gdy długa trajektoria pozwala na całkowitą przebudowę sceny.
* **Separacja zarysu od detali:** Trajektoria wyznacza stany pośrednie. W początkowych krokach ustala układ głównych elementów sceny, a w końcowych ($t \to 0$) generuje drobne szczegóły. Odpowiednie zarządzanie tymi etapami zapobiega zjawisku wizualnego przeładowania (tzw. *overbaking*).
* **Redukcja artefaktów i halucynacji:** Gładka, dobrze ukształtowana ścieżka zapobiega błędom strukturalnym. Model rzadziej generuje błędną anatomię (np. niepoprawną liczbę palców) lub niefizyczne połączenia obiektów.

## Narzędzia kontroli: Jak wpływamy na trajektorię?

Kontrola ścieżki *denoisingu* zachodzi na dwóch głównych etapach życia modelu i wykorzystuje ściśle określone mechanizmy, od modyfikacji architektury po nawigację w czasie rzeczywistym:

### 1. Training time
W fazie *training time* aktualizowane są *weights* modelu $\theta$, aby algorytm minimalizował *noise prediction loss*.
* **Loss function & Regularization:** *Dataset* i *loss function* projektuje się tak, aby wymusić proste ścieżki. Dodatkowe kary matematyczne zapobiegają niefizycznym załamaniom trajektorii.
* **Distillation:** Uczenie mniejszego modelu (tzw. *student model*) naśladowania stabilnej trajektorii większego modelu (*teacher model*) w celu redukcji niezbędnych kroków.
* **Control Adapters (np. ControlNet) & LoRA:** Dodatkowe sieci neuronowe lub celowane modyfikacje *weights*, które twardo zawężają trajektorię do zadanej geometrii (np. map głębi) lub konkretnego stylu malarskiego, nie niszcząc bazowej wiedzy modelu.

### 2. Inference time
W fazie generacji użytkownik modyfikuje przebieg fizycznej trajektorii w *latent space* korzystając z zamrożonego modelu:
* **Scheduler & Sampling Steps ($N$):** Wybór algorytmu rozwiązującego równania (np. Euler, DDIM) dopasowanego do krzywizny trajektorii oraz ustalenie gęstości jej podziału na kroki.
* **CFG Scale ($w$) & Negative Prompting:** Skalowanie siły, z jaką trajektoria podąża za warunkiem $c$. Dodatkowo, *negative prompt* matematycznie odpycha ścieżkę od niepożądanych rejonów (np. błędnej anatomii).
* **Denoising Strength:** Używany przy modyfikacji istniejących obrazów. Określa punkt startowy trajektorii ($t < T$) na osi czasu – decyduje, czy model zachowa oryginalną strukturę, czy zniszczy ją na rzecz wygenerowania nowej.
* **Prompt Scheduling & Regional Prompting:** Dynamiczna zmiana warunku tekstowego w trakcie pętli (np. budowa zarysu w pierwszych 50% kroków z jednym promptem, a detali w pozostałych z innym) lub wytyczanie osobnych trajektorii dla różnych stref obrazu i ich spajanie (tzw. *blending*).

## SDEdit w praktyce (SDE SEGMENT)

**SDEdit** (*Stochastic Differential Equations editing framework*) to bezpośredni przykład wykorzystania sterowania trajektorią do zarządzania procesem generacji. W odróżnieniu od klasycznych modeli generujących obrazy losowo od zera (*text-to-image*), narzędzie to służy przede wszystkim do intuicyjnej modyfikacji i syntezy obrazu na bazie odręcznego szkicu.

### Przebieg procesu (Pipeline)

SDEdit nie wymaga specyficznego treningu (*zero-shot*) – działa na zamrożonych wagach modelu. Główną ideą jest wejście w ścieżkę generacji w odpowiednim momencie. Nasz pierwotny wprowadzony przez użytkownika szkic lub uboższy w detale obraz staje się punktem startowym:

1. **Dodawanie szumu (Forward SDE):** Ręczny szkic rzadko wygląda realistycznie – jest pełen nienaturalnych krawędzi i płaskich plam koloru. Algorytm wstrzykuje do niego *noise* wykorzystując stochastyczne równania różniczkowe (SDE), zatrzymując proces w określonym momencie $t_0$. Szum niszczy artefakty nałożone odręcznie przez użytkownika, ale zachowuje globalny zarys obiektu (np. paletę barw i proporcje). W matematyce definiuje to klasyczne już równanie (dyfuzji w czasie ciągłym):
   $$ dx = f(x, t) dt + g(t) dw $$
   Gdzie $x$ to wektor obrazu, $f$ wyznacza kierunek średniego rozpadu, a $g(t)dw$ dodaje do danych odpowiednio skalowany szum.

2. **Odszumianie i Synteza (Reverse SDE):** Po osiagnięciu punktu $t_0$ model dyfuzyjny przejmuje stery i zaczyna *denoising*. Odwraca równanie różniczkowe, "wygładzając" obraz i rzutując go z powrotem na przestrzeń realistycznych reprezentacji wizualnych (tzw. *manifold projection*):
   $$ dx = [f(x, t) - g(t)^2 \nabla_x \log p_t(x)] dt + g(t) d\bar{w} $$
   Termin $\nabla_x \log p_t(x)$ określa się jako *score function*. To właśnie go optymalizuje sieć neuronowa podczas uczenia. W ten sposób algorytm generuje fotorealistyczny obraz, którego forma "budowa" opiera się na szkicu i dąży do zachowania jak najbliższego podobieństwa.

### Punkt wejścia ($t_0$)

Sednem całej techniki jest ustalenie idealnego poziomu ingerencji, czyli punktu wejścia na trajektorię ($t_0 \in [0, 1]$):
* **Zbyt duży $t_0$ (dużo szumu):** Oryginalna informacja podyktowana szkicem ulega całkowitemu i bezpowrotnemu zatarciu. Otrzymamy ładny i realistyczny wygenerowany obraz, ale w ogóle niezwiązany z wyjściowym pomysłem.
* **Zbyt mały $t_0$ (mało szumu):** Model nie ma wystarczającego "rozbiegu" (bardzo krótka odwrócona trajektoria), żeby naprawić niefizyczne krawędzie. Zachowamy doskonałą lojalność względem szkicu, ale otrzymamy równie amatorski efekt końcowy przypominający grafikę z programu graficznego.
* **Sweet spot (optymalny kompromis):** Starannie dobrany punkt $t_0$ idealnie znosi wady pierwszych dwóch metod, wyznaczając doskonały *trade-off* pomiędzy wiernością układu wyjściowego a abstrakcyjnym realizmem i szczegółowością modyfikowanej tekstury.

### Przykłady zastosowania (Demo Time)

Żeby zobaczyć na własne oczy, jak potężne jest to narzędzie i dlaczego samo wejście w odpowiednim momencie na trajektorię (bez konieczności dodatkowego trenowania modelu) daje tak niesamowite efekty, przygotowaliśmy dwa interaktywne przykłady:

#### Przykład 1: Generowanie abstrakcyjnych krajobrazów

Wykorzystamy czysty wejściowy obraz zbudowany z prostych plam kolorów – możemy go potraktować jako zwykły szkic wykonany pędzlem w Paincie.
* **Co robimy:** Za pomocą małego wbudowanego edytora narysuj prosty zarys rzeki oraz gór wokół ułożonych pod odpowiednim kątem. To będą nasze "bazowe" abstrakcyjne plamy.
* **Dlaczego to działa:** Algorytm przyjmuje ten ręcznie narysowany krajobraz, dodaje do niego sporo szumu (wchodzimy w trajektorię w punkcie $t_0$, powiedzmy, koło $0.7$), a model następnie go wygładza, zachowując kontury masywów i cieków wodnych, przekształcając płaskie plamy w fotorealistyczny, przepiękny widok.

#### Przykład 2: Edycja twarzy (Zbiór CelebA)

Tutaj SDEdit posłuży jako potężne narzędzie do edycji już istniejących zdjęć wysokiej jakości.
* **Co robimy:** Pobieramy gotowe zdjęcie z popularnego zestawu danych z twarzami celebrytów CelebA. Za pomocą edytora bezpośrednio domalowujemy mu np. wielkie, czarne wąsy, albo okulary przeciwsłoneczne, znowu dość abstrakcyjną, płaską "plamą".
* **Dlaczego to działa:** W tym przypadku dobieramy mniejsze $t_0$ (około $0.3 - 0.4$), ponieważ nie chcemy stracić tożsamości oryginalnej osoby. System musi dostać tylko tyle szumu, by połączyć nieudolnie dorysowane, czarne wąsy ze strukturą skóry, wygładzić przejścia i wyostrzyć sztucznie namalowane włosie, wpasowując je w fizykę twarzy i oświetlenie. Zbyt mocne wejście w trajektorię sprawiłoby, że celebryta całkowicie by zniknął na rzecz wygenerowania jakiegoś innej, choć wciąż "wąsatej" osoby.

In [ ]:
# Przykładowy kod do przygotowania środowiska i odpalenia demo
# Wymaga doinstalowania `gradio` i np. `diffusers` do wytyczania trajektorii i denoisingu:
# !pip install gradio diffusers transformers accelerate

import gradio as gr
import torch
from diffusers import StableDiffusionImg2ImgPipeline
from PIL import Image

# Wczytujemy model dla SDEdit w kontekście image-to-image
device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", 
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

def generate_landscape(sketch_image, prompt, strength):
    """
    Przyjmuje odręczny szkic krajobrazu (river, mountains itp).
    Stosuje wejście w trajektorię SDEdit zależne od `strength`.
    """
    if sketch_image is None: return None
    # Konwertujemy obraz z canvas (słownik z Gradio -> pil)
    image = Image.fromarray(sketch_image["composite"]).convert("RGB").resize((512, 512))
    
    # Przebieg trajektorii (Reverse SDE z zadanego punktu szumu)
    result = pipe(prompt=prompt, image=image, strength=strength, guidance_scale=7.5).images[0]
    return result

def edit_celeb_face(face_image, prompt, strength):
    """
    Przyjmuje zdjęcie powiedzmy z CelebA z namalowanymi, płaskimi "wąsami"
    lub innymi atrybutami, a następnie wygładza je do realizmu.
    """
    if face_image is None: return None
    image = Image.fromarray(face_image["composite"]).convert("RGB").resize((512, 512))
    
    # Przebieg SDEdit przy niskim $t_0$, by zachowac twarz (strength ~ 0.3)
    result = pipe(prompt=prompt, image=image, strength=strength, guidance_scale=7.5).images[0]
    return result

# ----------------- UI -----------------

with gr.Blocks(title="SDEdit: Trajectory Guidance Demo") as demo:
    gr.Markdown("## OśCzasu: Jak my na tym wpływamy?")
    
    with gr.Tab("1. Abstrakcyjny Krajobraz (np. Rzeka i góry)"):
            with gr.Row():
                with gr.Column():
                    gr.Markdown("Wybierz prosty pędzel i narysuj abstrakcyjny kształt – rzekę, słońce, góry. Nasz algorytm dopasuje trajektorię na dość wysokim punkcie $t_0$.")
                    landscape_canvas = gr.ImageMask(brush=gr.Brush(colors=["#3b82f6", "#22c55e", "#fcd34d"]), format="png", label="Narysuj szkic tutaj!")
                    landscape_prompt = gr.Textbox(label="Prompt", value="a beautiful landscape with a river and mountains, highly detailed, photorealistic")
                    landscape_strength = gr.Slider(0.1, 1.0, value=0.75, label="Siła szumu (t_0) - Więcej = Dalej w dół SDE")
                    landscape_btn = gr.Button("Wygeneruj Pejzaż")
                with gr.Column():
                    landscape_output = gr.Image(label="Wynikowy Obraz")

            landscape_btn.click(
                fn=generate_landscape, 
                inputs=[landscape_canvas, landscape_prompt, landscape_strength], 
                outputs=landscape_output
            )

    with gr.Tab("2. Edycja Twarzy z celebrytą (CelebA)"):
        with gr.Row():
            with gr.Column():
                gr.Markdown("Załaduj jakiekolwiek zdjęcie celebryty. Najlepiej dorysować odręcznie np. pofalowane krawędzie, wąsy, inne atrybuty. Szum wyczyści krawędzie i przyłączy je do oryginalnej struktury.")
                celeb_canvas = gr.ImageMask(brush=gr.Brush(colors=["#000000", "#ff0000"]), format="png", label="Pobierz i namaluj detale tutaj!")
                celeb_prompt = gr.Textbox(label="Prompt", value="a photo of a person wearing a mustache, realistic photography, sharp focus")
                celeb_strength = gr.Slider(0.1, 1.0, value=0.35, label="Siła szumu (t_0) - Mniej = Lojalność wobec kształtu")
                celeb_btn = gr.Button("Wyretuszuj i wygładź")
            with gr.Column():
                celeb_output = gr.Image(label="Wynikowy Obraz")

        celeb_btn.click(
            fn=edit_celeb_face, 
            inputs=[celeb_canvas, celeb_prompt, celeb_strength], 
            outputs=celeb_output
        )

demo.launch(inline=True, height=800) # <- Odkomentuj by wystartować serwer w trakcie prezentacji.

Loading weights: 100%|██████████| 396/396 [00:00<00:00, 1750.92it/s]]
StableDiffusionSafetyChecker LOAD REPORT from: /home/bartosz/Desktop/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 196/196 [00:00<00:00, 9684.33it/s]8.20it/s]
CLIPTextModel LOAD REPORT from: /home/bartosz/Desktop/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | 

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


100%|██████████| 27/27 [00:01<00:00, 19.67it/s]
Potential NSFW content was detected in one or more images. A black image will be returned instead. Try again with a different prompt and/or seed.
100%|██████████| 21/21 [00:01<00:00, 19.37it/s]
